# M9 · Cost y latency · Prompt caching + model routing

> Aplicas las dos optimizaciones que más ahorro real producen en LLM apps: **prompt caching** (hasta un 90% de descuento en los tokens cacheados) y **model routing** (un modelo nano clasifica; nano, mini o frontier ejecutan según la complejidad). Medimos antes y después con un mini-dashboard y, opcionalmente, subes las métricas a Langfuse.

## Qué aprendemos

1. **La fórmula del coste** · tokens de entrada (normales y cacheados) y de salida por su precio.
2. **OpenAI prompt caching** automático (a partir de 1024 tokens de prefijo idéntico).
3. **Anthropic prompt caching** (opcional) con cache breakpoints explícitos.
4. **Model routing por clasificador** · nano → mini → frontier.
5. **Medir el ahorro real** · tokens consumidos, tokens cacheados y latencia.
6. (Opcional) **Subir métricas a Langfuse** para verlas en el dashboard de M7.

## Por qué importa

Una LLM app con 100k requests/mes, system prompt de 4k tokens y respuestas de ~500 tokens:

- Sin optimizar con `gpt-5-mini`: ~200 USD/mes.
- Sin optimizar con `gpt-5.2`: ~1.400 USD/mes.
- Con caching + routing: fácilmente la mitad de coste, y menos latencia en los prompts largos.

A escala, esto paga ingenieros.

## Conecta con

- **clase-3 M1 · LLMOps** · el self-host (Ollama, vLLM) es la otra palanca de coste.
- **clase-3 M2 · Fine-tuning** · el break-even entre fine-tuning y prompting decide cuándo entrenar.
- **M7 · Observabilidad** · sin trazas no mides el ahorro: medir va antes de optimizar.
- **M8 · Safety** · cada guardrail añade llamadas: también tiene coste y latencia.

## Stack

`openai` · `anthropic` (opcional) · `langfuse>=4.0` (opcional).

## Hardware

Cualquier máquina · todo es API. Coste estimado: ~0,10 USD. Tiempo: ~5 min de principio a fin.

## Qué necesitas

- `OPENAI_API_KEY`.
- (Opcional) `ANTHROPIC_API_KEY` para la demo de cache breakpoints explícitos.
- (Opcional) Claves de Langfuse de M7 para subir la comparativa.

## Guion del lab

1. **Setup**.
2. **Tabla de precios** y fórmula del coste.
3. **El workload**: 8 preguntas variadas (fáciles y difíciles).
4. **Baseline**: todo a `gpt-5-mini` con un system prompt corto. Medimos coste y latencia.
5. **OpenAI prompt caching**: un system prompt largo y repetido activa la caché. Medimos el ahorro.
6. **Anthropic prompt caching** (si hay API key): el mismo patrón con cache breakpoints explícitos.
7. **Model routing**: clasificador barato + nano/mini/frontier, comparado con mandarlo todo a frontier.
8. **Comparativa final**: coste, latencia y % de ahorro de cada estrategia.
9. **(Opcional) Langfuse**: un trace por estrategia, visible en el dashboard.
10. **Recap y cierre del curso**.

## 1. Setup


In [1]:
%pip install -q "openai>=2" anthropic "langfuse>=4.0,<5"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from getpass import getpass


def cargar_clave(nombre: str, opcional: bool = False) -> bool:
    """Carga la clave del entorno, de los secretos de Colab o a mano. Devuelve True si queda cargada."""
    if os.environ.get(nombre):
        print(f"{nombre}: OK (entorno).")
        return True
    try:
        from google.colab import userdata
        v = userdata.get(nombre)
        if v:
            os.environ[nombre] = v
            print(f"{nombre}: OK (secretos de Colab).")
            return True
    except Exception:
        pass
    try:
        aviso = " (opcional, Enter para saltar)" if opcional else ""
        valor = getpass(f"Introduce tu {nombre}{aviso}: ").strip()
    except Exception:
        valor = ""  # entorno sin entrada interactiva
    if valor:
        os.environ[nombre] = valor
        print(f"{nombre}: OK (entrada manual).")
        return True
    if not opcional:
        raise RuntimeError(f"Falta {nombre}: es obligatoria para este lab.")
    print(f"{nombre}: no disponible, se omite la demo correspondiente.")
    return False


tiene_openai = cargar_clave("OPENAI_API_KEY")
tiene_anthropic = cargar_clave("ANTHROPIC_API_KEY", opcional=True)
tiene_langfuse = cargar_clave("LANGFUSE_PUBLIC_KEY", opcional=True) and cargar_clave("LANGFUSE_SECRET_KEY", opcional=True)
if tiene_langfuse:
    os.environ.setdefault("LANGFUSE_BASE_URL", os.environ.get("LANGFUSE_HOST", "https://cloud.langfuse.com"))

OPENAI_API_KEY: OK (entorno).
ANTHROPIC_API_KEY: no disponible, se omite la demo correspondiente.
LANGFUSE_PUBLIC_KEY: no disponible, se omite la demo correspondiente.


## 2. Tabla de precios y fórmula del coste

Precios en USD por **1M tokens** (referencia octubre 2026). Se usan para estimar el coste; comprueba siempre el pricing oficial antes de presupuestar en serio.

La fórmula es siempre la misma:

```
coste = (input_no_cacheado × precio_in + input_cacheado × precio_cacheado + output × precio_out) / 1.000.000
```

Dos detalles que sorprenden:

- El **output** cuesta entre 5 y 8 veces más que el input. En los modelos con razonamiento (familia GPT-5), los tokens de razonamiento se facturan como output aunque no los veas.
- El **input cacheado** cuesta un 10% del normal (90% de descuento) tanto en la familia GPT-5 como en Claude. En Anthropic, además, **escribir** en la caché cuesta un 25% más que el input normal.

In [3]:
PRECIOS = {
    # OpenAI (el input cacheado cuesta el 10% del input normal)
    "gpt-5.2":    {"in": 1.75, "out": 14.00, "in_cached": 0.175},
    "gpt-5-mini": {"in": 0.25, "out": 2.00,  "in_cached": 0.025},
    "gpt-5-nano": {"in": 0.05, "out": 0.40,  "in_cached": 0.005},
    # Anthropic (lectura de caché al 10%; escritura de caché al 125%)
    "claude-sonnet-4-6": {"in": 3.00, "out": 15.00, "in_cached": 0.30, "in_cache_write": 3.75},
    "claude-haiku-4-5":  {"in": 1.00, "out": 5.00,  "in_cached": 0.10, "in_cache_write": 1.25},
}


def coste(modelo: str, in_tokens: int, out_tokens: int, in_cached: int = 0) -> float:
    """Coste en USD. in_tokens incluye los cacheados (así lo cuenta OpenAI en prompt_tokens)."""
    p = PRECIOS.get(modelo)
    if not p:
        return 0.0
    in_no_cacheado = max(in_tokens - in_cached, 0)
    return (
        in_no_cacheado * p["in"]
        + in_cached * p["in_cached"]
        + out_tokens * p["out"]
    ) / 1_000_000


# Comprobación rápida.
print(f"1M de input + 1M de output con gpt-5-mini: ${coste('gpt-5-mini', 1_000_000, 1_000_000):.2f}")
print(f"Lo mismo con todo el input cacheado:       ${coste('gpt-5-mini', 1_000_000, 1_000_000, 1_000_000):.2f}")

1M de input + 1M de output con gpt-5-mini: $2.25
Lo mismo con todo el input cacheado:       $2.02


## 3. El workload

8 preguntas variadas: algunas fáciles (un nano las resuelve) y otras difíciles (necesitan un modelo más potente). Las usaremos en todas las estrategias.

In [4]:
WORKLOAD = [
    # Fáciles
    ("facil",  "¿Cuántos días tiene una semana? Responde en una frase."),
    ("facil",  "Convierte 100 USD a EUR con un tipo de 1 USD = 0,92 EUR."),
    ("facil",  "¿Cuál es la capital de Francia?"),
    ("facil",  "Saluda en español y pregunta cómo puedes ayudar, en una frase."),
    # Medias
    ("media",  "Resume en 3 viñetas qué es la caché de prompts y para qué sirve."),
    ("media",  "Explica qué es Retrieval-Augmented Generation en 4 frases."),
    # Difíciles
    ("dificil", "Diriges un equipo de 8 ingenieros en una empresa SaaS. Diseña un plan de 90 días para introducir evals automáticas en el ciclo de release, con riesgos, KPI y un cronograma realista."),
    ("dificil", "Compara tres estrategias de routing de modelos (cascade, clasificador explícito, bandit sensible al coste) con pros, contras y cuándo elegir cada una. Da pseudocódigo de la mejor para un chatbot de soporte."),
]

print(f"Workload: {len(WORKLOAD)} preguntas.")
for c, p in WORKLOAD:
    print(f"  [{c}] {p[:80]}")

Workload: 8 preguntas.
  [facil] ¿Cuántos días tiene una semana? Responde en una frase.
  [facil] Convierte 100 USD a EUR con un tipo de 1 USD = 0,92 EUR.
  [facil] ¿Cuál es la capital de Francia?
  [facil] Saluda en español y pregunta cómo puedes ayudar, en una frase.
  [media] Resume en 3 viñetas qué es la caché de prompts y para qué sirve.
  [media] Explica qué es Retrieval-Augmented Generation en 4 frases.
  [dificil] Diriges un equipo de 8 ingenieros en una empresa SaaS. Diseña un plan de 90 días
  [dificil] Compara tres estrategias de routing de modelos (cascade, clasificador explícito,


## 4. Baseline: todo a gpt-5-mini

Sin caching, sin routing, sin nada. Cada pregunta llama a `gpt-5-mini`. Medimos:

- tokens de input y output (los devuelve la API en `response.usage`),
- tokens cacheados (`usage.prompt_tokens_details.cached_tokens`),
- tiempo de pared (latencia),
- coste estimado.

System prompt **corto** (muy por debajo de 1024 tokens): aquí no puede haber caché.

En todas las llamadas usamos `reasoning_effort="low"` y `verbosity="low"`: en la familia GPT-5 los tokens de razonamiento se facturan como output, así que bajar el esfuerzo de razonamiento es en sí mismo una palanca de coste y latencia.

In [5]:
import time
from openai import OpenAI

cliente = OpenAI()

SYSTEM_CORTO = "Eres un asistente conciso en español."


# Dos palancas de coste de la familia GPT-5: menos razonamiento y respuestas más concisas.
# Los tokens de razonamiento se facturan como output, así que "low" abarata y acelera.
AJUSTES_POR_DEFECTO = {"reasoning_effort": "low", "verbosity": "low"}


def llamar_openai(modelo: str, system: str, user: str, **extra) -> dict:
    """Llama a OpenAI y devuelve un dict con la respuesta y las métricas."""
    extra = {**AJUSTES_POR_DEFECTO, **extra}
    t0 = time.perf_counter()
    try:
        r = cliente.chat.completions.create(
            model=modelo,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            **extra,
        )
        dt = time.perf_counter() - t0
        in_t = r.usage.prompt_tokens          # incluye los tokens cacheados
        out_t = r.usage.completion_tokens     # incluye los tokens de razonamiento
        detalles = r.usage.prompt_tokens_details
        in_cached = (detalles.cached_tokens or 0) if detalles else 0
        return {
            "ok": True,
            "respuesta": r.choices[0].message.content,
            "in_tokens": in_t,
            "out_tokens": out_t,
            "in_cached": in_cached,
            "latencia": dt,
            "modelo": modelo,
            "coste": coste(modelo, in_t, out_t, in_cached),
            "coste_sin_cache": coste(modelo, in_t, out_t, 0),
        }
    except Exception as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}", "modelo": modelo,
                "latencia": time.perf_counter() - t0, "in_tokens": 0, "out_tokens": 0,
                "in_cached": 0, "coste": 0.0, "coste_sin_cache": 0.0}


def ejecutar(workload, llamada_fn, etiqueta: str) -> dict:
    """Ejecuta el workload con la función dada. llamada_fn recibe (categoria, pregunta)."""
    print(f"\n=== Estrategia: {etiqueta} ===")
    filas = []
    t_total_0 = time.perf_counter()
    for cat, p in workload:
        r = llamada_fn(cat, p)
        filas.append({"categoria": cat, "pregunta": p[:50], **r})
        marca = "OK" if r.get("ok") else "ERR"
        print(f"  [{marca}] {r['modelo']:18s} in={r['in_tokens']:5d} cached={r['in_cached']:5d} "
              f"out={r['out_tokens']:5d} t={r['latencia']:6.2f}s ${r['coste']:.5f}")
        if not r.get("ok"):
            print(f"        {r['error'][:150]}")
    resumen = {
        "etiqueta": etiqueta,
        "filas": filas,
        "coste_total": sum(f["coste"] for f in filas),
        "coste_sin_cache": sum(f.get("coste_sin_cache", f["coste"]) for f in filas),
        "in_tok": sum(f["in_tokens"] for f in filas),
        "out_tok": sum(f["out_tokens"] for f in filas),
        "cached": sum(f["in_cached"] for f in filas),
        "wall": time.perf_counter() - t_total_0,
    }
    print(f"  --- TOTAL: ${resumen['coste_total']:.5f} | in={resumen['in_tok']} cached={resumen['cached']} "
          f"out={resumen['out_tok']} | wall={resumen['wall']:.1f}s")
    return resumen


def llamada_baseline(cat, p):
    return llamar_openai("gpt-5-mini", SYSTEM_CORTO, p)


resumen_baseline = ejecutar(WORKLOAD, llamada_baseline, "baseline (mini, prompt corto)")


=== Estrategia: baseline (mini, prompt corto) ===
  [OK] gpt-5-mini         in=   34 cached=    0 out=   15 t=  1.31s $0.00004
  [OK] gpt-5-mini         in=   41 cached=    0 out=   23 t=  1.16s $0.00006
  [OK] gpt-5-mini         in=   28 cached=    0 out=   12 t=  1.81s $0.00003
  [OK] gpt-5-mini         in=   34 cached=    0 out=   81 t=  1.56s $0.00017
  [OK] gpt-5-mini         in=   39 cached=    0 out=   99 t=  2.25s $0.00021
  [OK] gpt-5-mini         in=   33 cached=    0 out=   97 t=  1.74s $0.00020
  [OK] gpt-5-mini         in=   66 cached=    0 out= 1373 t= 11.96s $0.00276
  [OK] gpt-5-mini         in=   66 cached=    0 out=  845 t=  7.24s $0.00171
  --- TOTAL: $0.00518 | in=341 cached=0 out=2545 | wall=29.0s


## 5. OpenAI prompt caching automático

OpenAI cachea automáticamente el **prefijo** del prompt cuando supera los 1024 tokens y se repite idéntico en llamadas cercanas (en bloques de 128 tokens). No hay que tocar nada del código: solo **ordenar** el prompt con lo estable al principio (system prompt, ejemplos, documentación) y lo variable al final (la pregunta del usuario).

Simulamos un **system prompt largo** (en una app real sería tu manual de instrucciones + políticas + ejemplos), de unos 2.000 tokens. Pasamos además `prompt_cache_key`, que ayuda a OpenAI a enrutar las peticiones con el mismo prefijo a la misma caché.

1. Una llamada de **calentamiento** escribe el prefijo en la caché.
2. Lanzamos el workload: cada petición debería leer el prefijo de la caché.
3. Comparamos el coste real con el que habríamos pagado **sin** caché (mismos tokens, precio normal).

In [6]:
# Manual de soporte de una tienda online: lo estable va primero en el prompt.
POLITICAS = [
    "Reembolsos: validar ticket, validar fecha de compra (máximo 30 días), validar que el producto no se ha usado y aprobar o escalar al equipo de pagos.",
    "Cancelaciones: localizar el pedido, verificar su estado, cancelar solo si el estado es 'pendiente' o 'preparando' y enviar email de confirmación.",
    "Soporte técnico: identificar la versión de la app, el error exacto y los logs disponibles; abrir ticket con prioridad según el impacto.",
    "Envíos internacionales: informar de aranceles, plazos por zona (UE 3-5 días, resto 7-15 días) y número de seguimiento.",
    "Facturación: emitir factura con NIF del cliente; las rectificativas solo las emite el equipo de finanzas.",
    "Privacidad (RGPD): nunca compartir datos de otros clientes; las solicitudes de acceso o borrado se derivan al DPO.",
    "EU AI Act: indicar siempre al cliente que está hablando con un asistente de IA cuando lo pregunte.",
    "Accesibilidad (WCAG 2.2): ofrecer formatos alternativos y lenguaje claro cuando el cliente lo pida.",
    "Calidad de servicio: medimos CSAT, NPS y CES; cierra cada conversación preguntando si la duda quedó resuelta.",
    "SLA interno: primera respuesta en menos de 2 horas laborables; resolución en menos de 48 horas.",
]

SYSTEM_LARGO = (
    "Eres un asistente experto en operaciones de una empresa de e-commerce. "
    "Responde siempre en español, con tono profesional y el formato corporativo: "
    "una línea de saludo, un bloque de respuesta directa y una línea de cierre con una acción sugerida. "
    "Si la pregunta excede tu conocimiento, di 'consulta con tu manager o con el equipo legal'. "
    "Nunca inventes políticas internas.\n\n"
    "## Manual de políticas\n"
    + "\n".join(
        f"- [{seccion}.{i}] {politica}"
        for seccion in range(1, 6)  # 5 secciones del manual: pasamos holgadamente de 1024 tokens
        for i, politica in enumerate(POLITICAS, 1)
    )
)

CLAVE_CACHE = "lab-m9-soporte"


def llamada_cache(cat, p):
    return llamar_openai("gpt-5-mini", SYSTEM_LARGO, p, prompt_cache_key=CLAVE_CACHE)


# Calentamiento: una llamada escribe el prefijo en la caché.
calentamiento = llamada_cache("facil", "Hola")
print(f"Calentamiento: system prompt de {calentamiento['in_tokens']} tokens de input, cacheados={calentamiento['in_cached']}")
time.sleep(2)  # margen para que la caché quede disponible

resumen_cache = ejecutar(WORKLOAD, llamada_cache, "openai prompt caching")

ratio_cache = resumen_cache["cached"] / max(resumen_cache["in_tok"], 1)
ahorro_cache = 1 - resumen_cache["coste_total"] / max(resumen_cache["coste_sin_cache"], 1e-9)
print(f"\nTokens de input servidos desde caché: {ratio_cache:.0%}")
print(f"Coste del workload sin caché (estimado): ${resumen_cache['coste_sin_cache']:.5f}")
print(f"Coste real con caché:                    ${resumen_cache['coste_total']:.5f}  (ahorro {ahorro_cache:.0%})")

Calentamiento: system prompt de 1803 tokens de input, cacheados=0

=== Estrategia: openai prompt caching ===
  [OK] gpt-5-mini         in= 1816 cached= 1664 out=  154 t=  2.06s $0.00039
  [OK] gpt-5-mini         in= 1823 cached= 1664 out=  237 t=  2.81s $0.00056
  [OK] gpt-5-mini         in= 1810 cached= 1664 out=  226 t=  2.37s $0.00053
  [OK] gpt-5-mini         in= 1816 cached= 1664 out=  487 t=  5.06s $0.00105
  [OK] gpt-5-mini         in= 1821 cached= 1664 out=  246 t=  3.55s $0.00057
  [OK] gpt-5-mini         in= 1815 cached= 1664 out=  327 t=  3.99s $0.00073
  [OK] gpt-5-mini         in= 1848 cached= 1792 out= 1489 t= 13.87s $0.00304
  [OK] gpt-5-mini         in= 1848 cached= 1792 out= 1226 t=  9.70s $0.00251
  --- TOTAL: $0.00938 | in=14597 cached=13568 out=4392 | wall=43.4s

Tokens de input servidos desde caché: 93%
Coste del workload sin caché (estimado): $0.01243
Coste real con caché:                    $0.00938  (ahorro 25%)


### Qué esperamos ver

- `in` alto en cada petición (el system prompt sigue siendo largo) y `cached` casi igual a `in`: el prefijo viene de la caché.
- El ahorro se concentra en el **input**: los tokens cacheados cuestan el 10%. El output (incluido el razonamiento) se paga entero, por eso en preguntas con respuestas largas el ahorro total es menor.
- En sistemas reales con prompts de 4-8k tokens (RAG, few-shot, manuales) el ahorro es muy notable, y también baja la latencia del primer token.

> Nota: la caché de OpenAI vive normalmente entre 5 y 10 minutos de inactividad (hasta 1 hora en horas valle; `prompt_cache_retention="24h"` la extiende en modelos compatibles). Si pasan minutos largos entre llamadas, se enfría.

## 6. Anthropic prompt caching (cache breakpoints explícitos)

Anthropic exige marcar **explícitamente** qué parte del prompt cachear con `cache_control: {"type": "ephemeral"}`. Eso lo hace más controlable: decides dónde termina el bloque cacheado. La respuesta separa `cache_creation_input_tokens` (escritura, +25%), `cache_read_input_tokens` (lectura, -90%) e `input_tokens` (resto).

Ojo: el mínimo cacheable depende del modelo (en Haiku 4.5 son 4096 tokens), así que alargamos el system prompt.

Si no tienes API key de Anthropic, esta celda se salta.

In [7]:
resumen_anthropic = None
MODELO_ANT = "claude-haiku-4-5"

if tiene_anthropic:
    from anthropic import Anthropic

    cliente_ant = Anthropic()
    SYSTEM_ANT = "\n\n".join([SYSTEM_LARGO] * 3)  # Haiku 4.5 necesita >= 4096 tokens para cachear

    def llamada_anthropic_cache(cat, p):
        t0 = time.perf_counter()
        try:
            r = cliente_ant.messages.create(
                model=MODELO_ANT,
                max_tokens=1024,
                system=[{
                    "type": "text",
                    "text": SYSTEM_ANT,
                    "cache_control": {"type": "ephemeral"},  # <- el breakpoint
                }],
                messages=[{"role": "user", "content": p}],
            )
            u = r.usage
            leidos = u.cache_read_input_tokens or 0
            escritos = u.cache_creation_input_tokens or 0
            precio = PRECIOS[MODELO_ANT]
            c = (u.input_tokens * precio["in"] + escritos * precio["in_cache_write"]
                 + leidos * precio["in_cached"] + u.output_tokens * precio["out"]) / 1_000_000
            total_in = u.input_tokens + escritos + leidos
            return {"ok": True, "respuesta": r.content[0].text if r.content else "",
                    "in_tokens": total_in, "out_tokens": u.output_tokens, "in_cached": leidos,
                    "latencia": time.perf_counter() - t0, "modelo": MODELO_ANT, "coste": c,
                    "coste_sin_cache": coste(MODELO_ANT, total_in, u.output_tokens)}
        except Exception as e:
            return {"ok": False, "error": f"{type(e).__name__}: {e}", "modelo": MODELO_ANT,
                    "latencia": time.perf_counter() - t0, "in_tokens": 0, "out_tokens": 0,
                    "in_cached": 0, "coste": 0.0, "coste_sin_cache": 0.0}

    calentamiento_ant = llamada_anthropic_cache("facil", "Hola")
    print("Calentamiento Anthropic:", "OK" if calentamiento_ant["ok"] else calentamiento_ant["error"][:200])
    resumen_anthropic = ejecutar(WORKLOAD, llamada_anthropic_cache, "anthropic prompt caching")
else:
    print("Sin ANTHROPIC_API_KEY: se omite la demo de Anthropic (es opcional).")
    print("En la comparativa final aparecerá como 'no medido'.")

Sin ANTHROPIC_API_KEY: se omite la demo de Anthropic (es opcional).
En la comparativa final aparecerá como 'no medido'.


## 7. Model routing: nano → mini → frontier

Idea: clasificamos la pregunta y la mandamos al modelo **más barato que la sepa resolver**.

Dos variantes clásicas:

- **Routing por clasificador**: un modelo nano clasifica la dificultad antes de responder (1 llamada extra muy barata).
- **Routing en cascada**: lanzamos nano primero; si la respuesta es manifiestamente mala (regex, longitud, autoevaluación), reintentamos con un modelo mayor.

Aquí implementamos el **clasificador**, que es el más didáctico. Usamos el mismo system prompt corto que el baseline para que la comparación sea justa, y sumamos el coste y la latencia **reales** del clasificador.

¿Contra qué se compara un router? La alternativa real en producción suele ser **mandarlo todo al modelo frontier** "por si acaso". Así que primero medimos esa referencia (todo a `gpt-5.2`) y después el router.

In [8]:
CLASIF_PROMPT = (
    "Clasifica la dificultad de la pregunta del usuario en una sola palabra: "
    "FACIL, MEDIA o DIFICIL.\n\n"
    "Criterios:\n"
    "- FACIL: respuesta corta, conocimiento general, sin razonamiento complejo\n"
    "- MEDIA: requiere explicar o estructurar (3-6 frases)\n"
    "- DIFICIL: planes, diseños, comparativas entre varias opciones, pseudocódigo o más de 8 frases\n\n"
    "Pregunta: {pregunta}\n\nRespuesta (FACIL / MEDIA / DIFICIL):"
)

MODELO_POR_NIVEL = {
    "FACIL":   "gpt-5-nano",
    "MEDIA":   "gpt-5-mini",
    "DIFICIL": "gpt-5.2",
}


def clasificar(pregunta: str) -> tuple[str, dict]:
    """Pide a nano que clasifique la dificultad. Devuelve (nivel, métricas de la llamada)."""
    r = llamar_openai(
        "gpt-5-nano",
        "Clasificas la dificultad de preguntas. Responde con una sola palabra.",
        CLASIF_PROMPT.format(pregunta=pregunta),
        reasoning_effort="minimal",  # clasificar no necesita razonar: más rápido y barato
    )
    texto = (r.get("respuesta") or "").upper()
    for nivel in ["FACIL", "MEDIA", "DIFICIL"]:
        if nivel in texto:
            return nivel, r
    return "MEDIA", r  # valor por defecto conservador


def llamada_routing(cat, p):
    """Clasifica y manda la pregunta al modelo elegido. Suma coste y tiempo del clasificador."""
    t0 = time.perf_counter()
    nivel, r_clasif = clasificar(p)
    modelo = MODELO_POR_NIVEL[nivel]
    r = llamar_openai(modelo, SYSTEM_CORTO, p)
    if not r.get("ok") and modelo == "gpt-5.2":
        print(f"    ({modelo} no accesible en tu cuenta -> gpt-5-mini)")
        r = llamar_openai("gpt-5-mini", SYSTEM_CORTO, p)
    r["coste"] += r_clasif["coste"]
    r["coste_sin_cache"] += r_clasif["coste_sin_cache"]
    r["latencia"] = time.perf_counter() - t0
    r["clasificacion"] = nivel
    return r


# Referencia: todo al modelo frontier.
resumen_frontier = ejecutar(WORKLOAD, lambda cat, p: llamar_openai("gpt-5.2", SYSTEM_CORTO, p), "todo a gpt-5.2 (frontier)")

resumen_routing = ejecutar(WORKLOAD, llamada_routing, "model routing (nano/mini/5.2)")
print("\nClasificación por pregunta:")
for f in resumen_routing["filas"]:
    print(f"  esperado={f['categoria']:8s} clasificado={f['clasificacion']:8s} -> {f['modelo']}")


=== Estrategia: todo a gpt-5.2 (frontier) ===
  [OK] gpt-5.2            in=   34 cached=    0 out=    9 t=  1.17s $0.00019
  [OK] gpt-5.2            in=   41 cached=    0 out=   18 t=  1.06s $0.00032
  [OK] gpt-5.2            in=   28 cached=    0 out=   10 t=  1.02s $0.00019
  [OK] gpt-5.2            in=   34 cached=    0 out=   12 t=  0.94s $0.00023
  [OK] gpt-5.2            in=   39 cached=    0 out=  118 t=  2.55s $0.00172
  [OK] gpt-5.2            in=   33 cached=    0 out=  123 t=  2.94s $0.00178
  [OK] gpt-5.2            in=   66 cached=    0 out= 2139 t= 34.61s $0.03006
  [OK] gpt-5.2            in=   66 cached=    0 out= 1223 t= 19.49s $0.01724
  --- TOTAL: $0.05172 | in=341 cached=0 out=3652 | wall=63.8s

=== Estrategia: model routing (nano/mini/5.2) ===
  [OK] gpt-5-nano         in=   34 cached=    0 out=   15 t=  2.37s $0.00002
  [OK] gpt-5-nano         in=   41 cached=    0 out=   94 t=  2.64s $0.00005
  [OK] gpt-5-nano         in=   28 cached=    0 out=   12 t=  1.75s $0

## 8. Comparativa final

Dos comparaciones justas:

- **Prompt corto**: todo a frontier, baseline (todo a mini) y routing (mismo system prompt, cambia el modelo). La referencia es "todo a frontier".
- **Prompt largo**: el mismo workload con y sin el descuento de caché (mismos tokens, distinto precio).

La columna `%` indica la diferencia de coste respecto a su referencia.

In [9]:
def fila(etiqueta, coste_total, wall, ref_coste, ref_wall=None):
    delta = (coste_total - ref_coste) / max(ref_coste, 1e-9)
    w = f"wall {wall:6.1f}s" if wall is not None else "wall      -"
    if ref_wall and wall is not None:
        w += f" ({(wall - ref_wall) / ref_wall * 100:+5.1f}%)"
    print(f"  {etiqueta:38s} | coste ${coste_total:.5f} ({delta * 100:+6.1f}%) | {w}")


print("=== Prompt corto: frontier vs baseline vs routing (referencia: todo a frontier) ===")
for r in [resumen_frontier, resumen_baseline, resumen_routing]:
    fila(r["etiqueta"], r["coste_total"], r["wall"], resumen_frontier["coste_total"], resumen_frontier["wall"])

print("\n=== Prompt largo: sin caché vs con caché ===")
fila("openai sin caché (estimado)", resumen_cache["coste_sin_cache"], None, resumen_cache["coste_sin_cache"])
fila(resumen_cache["etiqueta"], resumen_cache["coste_total"], resumen_cache["wall"], resumen_cache["coste_sin_cache"])
if resumen_anthropic is not None:
    fila("anthropic sin caché (estimado)", resumen_anthropic["coste_sin_cache"], None, resumen_anthropic["coste_sin_cache"])
    fila(resumen_anthropic["etiqueta"], resumen_anthropic["coste_total"], resumen_anthropic["wall"], resumen_anthropic["coste_sin_cache"])
else:
    print("  anthropic prompt caching               | no medido (falta ANTHROPIC_API_KEY)")

print("\n=== Tokens cacheados ===")
for r in [resumen_frontier, resumen_baseline, resumen_cache, resumen_routing] + ([resumen_anthropic] if resumen_anthropic else []):
    ratio = r["cached"] / max(r["in_tok"], 1)
    print(f"  {r['etiqueta']:38s} | in={r['in_tok']:6d} cached={r['cached']:6d} ({ratio:.0%}) out={r['out_tok']:6d}")

=== Prompt corto: frontier vs baseline vs routing (referencia: todo a frontier) ===
  todo a gpt-5.2 (frontier)              | coste $0.05172 (  +0.0%) | wall   63.8s ( +0.0%)
  baseline (mini, prompt corto)          | coste $0.00518 ( -90.0%) | wall   29.0s (-54.5%)
  model routing (nano/mini/5.2)          | coste $0.00597 ( -88.5%) | wall   43.5s (-31.9%)

=== Prompt largo: sin caché vs con caché ===
  openai sin caché (estimado)            | coste $0.01243 (  +0.0%) | wall      -
  openai prompt caching                  | coste $0.00938 ( -24.6%) | wall   43.4s
  anthropic prompt caching               | no medido (falta ANTHROPIC_API_KEY)

=== Tokens cacheados ===
  todo a gpt-5.2 (frontier)              | in=   341 cached=     0 (0%) out=  3652
  baseline (mini, prompt corto)          | in=   341 cached=     0 (0%) out=  2545
  openai prompt caching                  | in= 14597 cached= 13568 (93%) out=  4392
  model routing (nano/mini/5.2)          | in=   341 cached=     0 (0%) ou

### Cómo leer la tabla

- **Caching**: deberías ver `cached` cerca del 90-100% del input en `openai prompt caching`. Si sale 0, la caché no se calentó (espera unos segundos y repite la celda, o alarga el system prompt). El ahorro total depende de cuánto pesa el input frente al output: con prompts largos y respuestas cortas es enorme.
- **Caching best-effort**: alguna petición suelta puede no acertar en la caché (`cached=0`); es normal, OpenAI no lo garantiza.
- **Routing**: frente a "todo a frontier" el ahorro es grande, porque solo lo difícil llega al modelo caro. Frente a "todo a mini" puede **subir**: cada pregunta que el router manda a frontier cuesta varias veces más. El router no compite con el modelo barato en precio, sino con el caro en calidad. Revisa la columna de clasificación: un router que se equivoca degrada calidad (o gasta de más).
- **Latencia (wall)**: el clasificador añade una llamada en serie a cada pregunta (1-2 s), y el modelo frontier tarda más en lo difícil. El caching mejora sobre todo el tiempo hasta el primer token, que aquí no medimos aparte.

> En producción, mide siempre con tu workload real y con trazas (M7). Estos números son orientativos.

## 9. (Opcional) Subir las métricas a Langfuse

Si tienes claves de Langfuse (las de M7), creamos un trace por estrategia con los totales como metadata y tres scores numéricos (coste, latencia y ratio cacheado). Así ves la comparativa en el dashboard, no solo en un print. Si no hay claves, la celda se salta sin error.

In [10]:
if tiene_langfuse:
    from langfuse import get_client, propagate_attributes

    lf = get_client()
    try:
        conectado = lf.auth_check()
    except Exception as e:
        print(f"No se pudo conectar a Langfuse ({type(e).__name__}): revisa claves y LANGFUSE_BASE_URL.")
        conectado = False

    if conectado:
        def subir(resumen):
            if resumen is None:
                return
            try:
                with lf.start_as_current_observation(as_type="span", name=f"estrategia:{resumen['etiqueta']}") as sp:
                    with propagate_attributes(tags=["clase-5", "m9", "comparativa"]):
                        sp.update(metadata={
                            "in_tok": resumen["in_tok"],
                            "out_tok": resumen["out_tok"],
                            "cached": resumen["cached"],
                            "coste_total_usd": resumen["coste_total"],
                            "wall_seg": resumen["wall"],
                        })
                        sp.score_trace(name="coste_usd", value=resumen["coste_total"], data_type="NUMERIC")
                        sp.score_trace(name="wall_seg", value=resumen["wall"], data_type="NUMERIC")
                        sp.score_trace(name="ratio_cached", value=resumen["cached"] / max(resumen["in_tok"], 1), data_type="NUMERIC")
                print(f"Subido: {resumen['etiqueta']}")
            except Exception as e:
                print(f"No se pudo subir {resumen['etiqueta']}: {e}")

        for r in [resumen_frontier, resumen_baseline, resumen_cache, resumen_anthropic, resumen_routing]:
            subir(r)
        lf.flush()
        print("\nAbre Langfuse y filtra por tag m9 para ver la comparativa.")
else:
    print("Sin claves de Langfuse: se omite la subida (es opcional).")
    print("Configúralas como en M7 y vuelve a ejecutar esta celda si quieres verlo en el dashboard.")

Sin claves de Langfuse: se omite la subida (es opcional).
Configúralas como en M7 y vuelve a ejecutar esta celda si quieres verlo en el dashboard.


## 10. Recap y cierre del curso

### Qué has medido

- **Referencias**: todo a gpt-5.2 (frontier) y todo a gpt-5-mini, sin trucos.
- **OpenAI caching**: system prompt largo y estable al principio; la caché se activa sola.
- **(Opcional) Anthropic caching**: el mismo patrón con un cache breakpoint explícito.
- **Model routing**: nano clasifica; nano, mini o 5.2 ejecutan según la complejidad.

### Ideas clave

- **Coste = tokens × precio**, y el output (con razonamiento) suele dominar.
- **Mide antes de optimizar**: los % cambian mucho según el workload real.
- **Prompt caching** es la victoria más rápida: casi cero código, solo ordenar bien el prompt.
- **Model routing** exige disciplina y reevaluación continua: cuidado con degradar calidad.
- **Batch API** (no cubierta aquí) añade otro 50% de descuento si tu workload no es en tiempo real.
- **Streaming** mejora la latencia percibida sin tocar el coste.

### Retos para practicar

1. **Router en cascada**: lanza nano; si la respuesta es muy corta o contiene "no estoy seguro", relanza con mini.
2. **Clasificador sin LLM**: usa embeddings + similitud coseno contra preguntas tipo (aún más barato).
3. **Caché semántica de respuestas**: un dict en memoria con embeddings para no repreguntar lo mismo.
4. **Batch API real**: convierte el workload en un `.jsonl`, mándalo por la Batch API y compara coste y latencia.
5. **Break-even**: calcula con tus números cuándo un modelo pequeño fine-tuneado (clase-3) sale más barato que caching + routing.

### Fin del curso

Con M9 cierras el ciclo completo: de los fundamentos de los LLM al fine-tuning y la evaluación (clase-3), al RAG medido con RAGas (clase-4) y a los agentes en producción: construidos (M1-M6), observados (M7), protegidos (M8) y ahora optimizados en coste y latencia (M9). Ya tienes todas las piezas para llevar una LLM app de la idea a producción y mantenerla sana. ¡Enhorabuena!